1. Create Donut Shapefile from Project KML -> Zipped Shapefile

In [ ]:
import pandas as pd# Import necessary libraries
import geopandas as gpd
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import os
from pyproj import CRS
import zipfile
import glob
import plotly.graph_objects as go

# Set the path to your KML file - replace with your actual KML file path
kml_file_path = '/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/data/proj_shape/674_VCS_KML_0674_16122021.kml'

# Set output directory and filename (without extension)
output_dir = "/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/data/proj_shape/"
output_filename = "buffered_geometry_donut"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load the KML file
gdf = gpd.read_file(kml_file_path, driver='KML')

# Print original CRS information
print(f"Original CRS: {gdf.crs}")

# If the CRS is not set (common with KML files), assume it's EPSG:4326 (WGS84)
if gdf.crs is None:
    gdf.crs = CRS.from_epsg(4326)
    print("Set CRS to EPSG:4326 (WGS84)")

# Transform to Web Mercator (EPSG:3857)
gdf_web_mercator = gdf.to_crs(epsg=3857)
print(f"New CRS: {gdf_web_mercator.crs}")

# Buffer by 10km (10,000 meters in Web Mercator)
gdf_buffered = gdf_web_mercator.copy()
gdf_buffered['geometry'] = gdf_web_mercator.geometry.buffer(10000)

# Create the ring by subtracting the original geometry from the buffered geometry
gdf_ring = gdf_buffered.copy()
# Apply the difference operation for each geometry
gdf_ring['geometry'] = [buffered.difference(original) for buffered, original 
                        in zip(gdf_buffered.geometry, gdf_web_mercator.geometry)]

# Plot to visualize the results
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

# Plot original geometry
gdf_web_mercator.plot(ax=ax[0], color='blue', alpha=0.5)
ax[0].set_title('Original Geometry')

# Plot buffered geometry
gdf_buffered.plot(ax=ax[1], color='red', alpha=0.5)
ax[1].set_title('Buffered Geometry (10km)')

# Plot ring geometry (buffer minus original)
gdf_ring.plot(ax=ax[2], color='green', alpha=0.5)
ax[2].set_title('Ring Geometry (10km buffer minus original)')

# Add original geometry outline to all plots for reference
for i in range(3):
    gdf_web_mercator.boundary.plot(ax=ax[i], color='blue', linewidth=1)

plt.tight_layout()
plt.show()

# Print information about the geometries
print(f"Original area: {gdf_web_mercator.area.sum()} square meters")
print(f"Buffered area: {gdf_buffered.area.sum()} square meters")
print(f"Ring area: {gdf_ring.area.sum()} square meters")

# Save the buffered geometry as a shapefile
output_shapefile = os.path.join(output_dir, output_filename)
gdf_ring.to_file(output_shapefile, driver="ESRI Shapefile")
print(f"Saved buffered geometry to {output_shapefile}.shp (and related files)")

# Create a zip file containing all the shapefile components
zip_filename = f"{output_shapefile}.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Find all files related to the shapefile
    shapefile_components = glob.glob(f"{output_shapefile}.*")
    for file in shapefile_components:
        # Add file to the zip (with just the filename, not the full path)
        zipf.write(file, os.path.basename(file))
        print(f"Added {os.path.basename(file)} to zip archive")

print(f"All shapefile components zipped to {zip_filename}")

2. Upload to GEE and Process with : [Prod]_Class_TimeSeries_Moving_Avg -> CSV 

3. Load the csv and upload to supabase

In [10]:
# Import necessary libraries
import json
from supabase import create_client
import numpy as np
from dotenv import load_dotenv
import pandas as pd
import os
from typing import Dict, Any
from supabase import create_client, Client
import plotly.graph_objects as go

# Load environment variables
load_dotenv()

class LandUseTStoSupabasePipeline:
    def __init__(self, project_code: str, file_path: str):
        """Initialize the pipeline with Supabase credentials"""
        self.supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
        self.project_code = project_code 
        self.file_path = file_path
        self.df = pd.read_csv(self.file_path)
        
    def get_project_id(self) -> str:
        """Retrieve project_id from projects table using project_code"""
        try:
            response = (self.supabase.table("projects")
                       .select("id")
                       .eq("project_code", self.project_code)
                       .execute())
            if response.data and len(response.data) > 0:
                return response.data[0]["id"]
            print(f"No project found with project_code: {self.project_code}")
            return None
        except Exception as e:
            print(f"Error retrieving project_id: {str(e)}")
            return None
        
    def upload_to_supabase(self, project_id: str, table_name: str = "landuse_time_series"):
        """Upload land use time series to Supabase with project_id matching"""
        
        # Add project_id to all rows
        self.df["project_id"] = project_id
        
        columns = [
            'project_id', 'month', 'bare', 'built', 'crops', 
            'flooded_vegetation', 'grass', 'shrub_and_scrub', 
            'snow_and_ice', 'trees', 'water'
        ] 
        df_upload = self.df[columns]
        
        try:
            # First, delete existing records for this project to avoid duplicates
            delete_response = (self.supabase.table(table_name)
                              .delete()
                              .eq("project_id", project_id)
                              .execute())
            
            print(f"Deleted {len(delete_response.data)} existing records")
            
            # Then insert all new records
            data = df_upload.to_dict(orient='records')
            
            # Insert in batches if there are many records
            batch_size = 100
            for i in range(0, len(data), batch_size):
                batch = data[i:i+batch_size]
                response = self.supabase.table(table_name).insert(batch).execute()
                print(f"Inserted batch {i//batch_size + 1}/{(len(data) + batch_size - 1)//batch_size}")
            
            print(f"Successfully uploaded {len(data)} records for project {project_id}")
            return True
        
        except Exception as e:
            print(f"Error uploading to Supabase: {str(e)}")
            return None
        
    def plotting(self):        
        # Define columns to plot (exclude 'month' and 'system:index')
        value_columns = [
            'water', 
            'bare', 'built', 'snow_and_ice', 
            'grass',
            'crops',
            'shrub_and_scrub', 
            'flooded_vegetation', 
            'trees', 
            'null'
            ]

        # Create a stacked area chart using Plotly
        fig = go.Figure()

        # Add a trace for each value column
        for col in value_columns:
            fig.add_trace(
                go.Scatter(
                    x=self.df['month'],
                    y=self.df[col],
                    mode='lines',
                    name=col,
                    stackgroup='one',  # Stack the areas
                    line=dict(width=0.5)
                )
            )

        # Update layout for better visualization
        fig.update_layout(
            title='Monthly Pixel Counts by Land Cover Class (2024)',
            xaxis_title='Month',
            yaxis_title='Pixel Count',
            xaxis=dict(tickangle=45),
            legend_title='Class',
            hovermode='x unified',
            template='plotly_white',
            showlegend=True
        )

        # Display the chart (in a Jupyter notebook or similar environment)
        fig.show()

        # Optionally, save the chart as an HTML file for sharing
        fig.write_html('stacked_area_chart.html')
        
        return "plotted successfully"
    
    # Main function to run the pipeline
    def process_landuse_pipeline(self):
        """
        Run the complete pipeline to upload land use time series data.
        
        Parameters:
        - project_code: The project code to link the data to
        """
        # Check if the project code exists
        if not self.get_project_id():
            print(f"Error: Project code '{self.project_code}' does not exist in the projects table")
            return False
        
        # Upload the data
        success = self.upload_to_supabase(self.get_project_id())
        
        if success:
            print(f"Pipeline completed successfully for project code: {self.project_code}")
        else:
            print(f"Pipeline failed for project code: {self.project_code}")
        
        # plot the file 
        self.plotting()
        
        return success


In [12]:
# load data frame from csv
filepath = '/Users/beckyxu/Downloads/moving_avg_pixel_counts_2016_2025_res10_test.csv'
pipeline = LandUseTStoSupabasePipeline(project_code="001", file_path=filepath)
pipeline.process_landuse_pipeline()

Deleted 99 existing records
Inserted batch 1/1
Successfully uploaded 99 records for project 00000000-0000-0000-0000-000000000001
Pipeline completed successfully for project code: 001


True